# Gaussian Embeddings: Closed-Form Derivations

What happens to the deterministic contrastive loss and similarity when the embeddings are Gaussian?

We show that the **expected similarity** under Gaussian embeddings has an exact closed form
that recovers the Lemma 3.1 identity kernel, and that Monte Carlo estimation is unnecessary.

## Setup

**Deterministic case.** Given paired embeddings $\mathbf{z}_i, \mathbf{z}_j \in \mathbb{R}^D$ and binary label $y \in \{0,1\}$:

$$p_{ij} = \exp\!\bigl(-s\,\|\mathbf{z}_i - \mathbf{z}_j\|_2^2\bigr)$$

where $s$ is the scale parameter (default $s=1$; original paper uses $s=0.25$).

The loss is binary cross-entropy:

$$\mathcal{L} = -\frac{1}{N}\sum_n \bigl[y_n \log p_n + (1-y_n)\log(1-p_n)\bigr]$$

**Probabilistic case.** Now suppose embeddings are Gaussian:

$$\mathbf{z}_i \sim \mathcal{N}(\boldsymbol{\mu}_i,\, \text{diag}(\boldsymbol{\sigma}_i^2)), \qquad \mathbf{z}_j \sim \mathcal{N}(\boldsymbol{\mu}_j,\, \text{diag}(\boldsymbol{\sigma}_j^2))$$

Then their difference is also Gaussian:

$$\mathbf{z}_i - \mathbf{z}_j \sim \mathcal{N}(\boldsymbol{\delta},\, \text{diag}(\mathbf{v}))$$

where $\delta_d = \mu_{i,d} - \mu_{j,d}$ and $v_d = \sigma_{i,d}^2 + \sigma_{j,d}^2$.

We want the expected similarity $\mathbb{E}[p_{ij}]$ and the expected loss $\mathbb{E}[\mathcal{L}]$.

## Expected Similarity: $\mathbb{E}[p_{ij}]$

Since dimensions are independent, the expectation factorizes:

$$\mathbb{E}\bigl[\exp(-s\|\mathbf{z}_i - \mathbf{z}_j\|^2)\bigr] = \prod_{d=1}^{D} \mathbb{E}\bigl[\exp\bigl(-s(\delta_d + \epsilon_d)^2\bigr)\bigr]$$

where $\epsilon_d \sim \mathcal{N}(0, v_d)$. Each factor is a standard Gaussian integral.

### Per-dimension integral

For a single dimension $d$, write $\epsilon \sim \mathcal{N}(0, v)$:

$$I = \mathbb{E}\bigl[e^{-s(\delta + \epsilon)^2}\bigr] = \frac{1}{\sqrt{2\pi v}} \int_{-\infty}^{\infty} \exp\!\left(-s(\delta + x)^2 - \frac{x^2}{2v}\right) dx$$

Expand the exponent:

$$-s(\delta + x)^2 - \frac{x^2}{2v} = -s\delta^2 - 2s\delta x - \underbrace{\left(s + \frac{1}{2v}\right)}_{a}\, x^2$$

where $a = s + \frac{1}{2v} = \frac{2sv + 1}{2v}$.

Complete the square in $x$:

$$-a\!\left(x + \frac{s\delta}{a}\right)^{\!2} + \frac{s^2\delta^2}{a} - s\delta^2 = -a\!\left(x + \frac{s\delta}{a}\right)^{\!2} - s\delta^2\!\left(1 - \frac{s}{a}\right)$$

Since $\frac{a - s}{a} = \frac{1/(2v)}{(2sv+1)/(2v)} = \frac{1}{2sv+1}$, the exponent becomes:

$$-a\!\left(x + \frac{s\delta}{a}\right)^{\!2} - \frac{s\,\delta^2}{2sv + 1}$$

The Gaussian integral over the completed square gives $\sqrt{\pi / a}$:

$$I = \frac{1}{\sqrt{2\pi v}} \cdot \sqrt{\frac{\pi}{a}} \cdot \exp\!\left(-\frac{s\,\delta^2}{2sv + 1}\right) = \frac{1}{\sqrt{2\pi v}} \cdot \sqrt{\frac{2\pi v}{2sv + 1}} \cdot \exp\!\left(-\frac{s\,\delta^2}{2sv + 1}\right)$$

$$\boxed{I = \frac{1}{\sqrt{2sv + 1}}\,\exp\!\left(-\frac{s\,\delta^2}{2sv + 1}\right)}$$

### Full product over dimensions

Multiplying over all $D$ dimensions:

$$\mathbb{E}[p_{ij}] = \prod_{d=1}^{D} \frac{1}{\sqrt{2sv_d + 1}}\,\exp\!\left(-\frac{s\,\delta_d^2}{2sv_d + 1}\right)$$

Taking the log:

$$\boxed{\log\,\mathbb{E}[p_{ij}] = -\frac{1}{2}\sum_{d=1}^{D}\left[\log(2sv_d + 1) + \frac{2s\,\delta_d^2}{2sv_d + 1}\right]}$$

**Sanity check:** when $v_d = 0$ (no variance), each term reduces to $\log 1 + 2s\delta_d^2 = 2s\delta_d^2$,
giving $\log p = -s\sum_d \delta_d^2 = -s\|\boldsymbol{\delta}\|^2$, recovering the deterministic formula.

## Connection to Lemma 3.1

Lemma 3.1 (identity form, $K = \alpha I$) gives:

$$\log p_{ij}^{\text{Lemma}} = -\frac{1}{2}\sum_d\left[\log\!\left(\frac{v_d}{\alpha} + 1\right) + \frac{\delta_d^2}{v_d + \alpha}\right]$$

Setting this equal to $\log\,\mathbb{E}[p_{ij}]$ and matching term by term:

**Normalization term:**

$$\log\!\left(\frac{v_d}{\alpha} + 1\right) = \log(2sv_d + 1) \;\implies\; \frac{v_d}{\alpha} = 2sv_d \;\implies\; \alpha = \frac{1}{2s}$$

**Distance term:**

$$\frac{\delta_d^2}{v_d + \alpha} = \frac{2s\,\delta_d^2}{2sv_d + 1} = \frac{\delta_d^2}{v_d + 1/(2s)} \quad \checkmark$$

Both terms agree. Therefore:

$$\boxed{\mathbb{E}\bigl[\exp(-s\|\mathbf{z}_i - \mathbf{z}_j\|^2)\bigr] = p_{ij}^{\text{Lemma 3.1}}\Big|_{K = \frac{1}{2s}I}}$$

| Scale $s$ | Setting | $\alpha = 1/(2s)$ |
|-----------|---------|-------------------|
| 1.0 | Default | $\alpha = 0.5$ |
| 0.25 | Original paper ($d^2/4$) | $\alpha = 2.0$ |

**The probabilistic similarity formula (Lemma 3.1, identity form) IS the analytically exact
expected deterministic similarity under Gaussian embeddings.** Monte Carlo would only add noise.

## Expected Loss: $\mathbb{E}[\mathcal{L}]$

The BCE loss per pair is $\ell = -y\log p - (1-y)\log(1-p)$.

Under Gaussian embeddings, we can either:

- **Option A:** Compute $\mathbb{E}[\ell(p)]$ -- average the loss over Gaussian samples (what MC estimates)
- **Option B:** Compute $\ell(\mathbb{E}[p])$ -- plug the expected similarity into BCE

### Option A: $\mathbb{E}[\ell(p)]$ (MC target)

**Positive pairs** ($y = 1$):

$$\mathbb{E}[-\log p] = \mathbb{E}\bigl[s\|\mathbf{z}_i - \mathbf{z}_j\|^2\bigr] = s\sum_d \mathbb{E}\bigl[(\delta_d + \epsilon_d)^2\bigr] = s\sum_d (\delta_d^2 + v_d)$$

This is the **expected distance** form -- exact closed form.

**Negative pairs** ($y = 0$):

$$\mathbb{E}\bigl[-\log(1 - p)\bigr] = \mathbb{E}\bigl[-\log\bigl(1 - e^{-s\|\mathbf{z}_i - \mathbf{z}_j\|^2}\bigr)\bigr]$$

The composition $-\log(1 - e^{-s \cdot})$ applied to $\|\mathbf{z}_i - \mathbf{z}_j\|^2$ has **no closed form**
under a Gaussian distribution. This would require Monte Carlo or approximation.

### Option B: $\ell(\mathbb{E}[p])$ (plug-in, Lemma 3.1)

Since $\mathbb{E}[p]$ has a closed form (derived above), we can plug it directly into BCE:

**Positive pairs** ($y = 1$):

$$-\log\,\mathbb{E}[p] = \frac{1}{2}\sum_d\left[\log(2sv_d + 1) + \frac{2s\,\delta_d^2}{2sv_d + 1}\right]$$

**Negative pairs** ($y = 0$):

$$-\log\bigl(1 - \mathbb{E}[p]\bigr)$$

where $\mathbb{E}[p] = \prod_d (2sv_d+1)^{-1/2}\exp\!\left(-\frac{s\delta_d^2}{2sv_d+1}\right)$ is known.

Both terms are **fully closed-form**, deterministic, and cheap to compute.

### Relationship: Jensen's inequality

Both $-\log p$ and $-\log(1-p)$ are **convex** in $p$. By Jensen's inequality:

$$\ell\bigl(\mathbb{E}[p]\bigr) \;\leq\; \mathbb{E}\bigl[\ell(p)\bigr]$$

The plug-in loss (Option B / Lemma 3.1) is a **lower bound** on the MC loss (Option A).
The gap vanishes as $v_d \to 0$.

In practice this means Lemma 3.1 gives a slightly optimistic loss estimate compared to MC,
but the difference is negligible when variances are small relative to the mean separation.

## Summary

| Expression | Closed form? | Formula | MC needed? |
|-----------|:---:|---|:---:|
| $\mathbb{E}[p_{ij}]$ (similarity) | Yes | Lemma 3.1 with $\alpha = 1/(2s)$ | No |
| $\mathbb{E}[-\log p]$ (pos. loss) | Yes | $s\sum_d(\delta_d^2 + v_d)$ | No |
| $\mathbb{E}[-\log(1-p)]$ (neg. loss) | No | -- | Yes |
| $-\log\,\mathbb{E}[p]$ (pos. plug-in) | Yes | $\frac{1}{2}\sum_d[\log(2sv_d+1) + \frac{2s\delta_d^2}{2sv_d+1}]$ | No |
| $-\log(1 - \mathbb{E}[p])$ (neg. plug-in) | Yes | Evaluate from $\mathbb{E}[p]$ | No |

**Key result:** The Lemma 3.1 identity form with $K = \frac{1}{2s}I$ is the
exact closed-form expected deterministic similarity under Gaussian embeddings.
No Monte Carlo sampling is needed -- it would only add noise to an analytically known quantity.